In [2]:
import pandas as pd

In [3]:
df_annotate = pd.read_csv("../model with metadata1/dataset/thesis_full.csv")
df_annotate

,text_id,annotators_id,text,topic,is_noise_or_spam_text,toxicity,polarized,profanity_obscenity,threat_incitement_to_violence,insults,identity_attack,sexually_explicit,key,cleaned_text,processed_text
0,1-1,"['7', '15']",Wajah Jurnalis Dibalut Perban Saat Melaporkan ...,UNKNOWN,"['0', '0']","['0', '0']","['0', '0']","['0', '0']","['0', '0']","['0', '0']","['0', '0']","['0', '0']","1-1_['7', '15']",Wajah jurnalis dibalut perban saat melaporkan ...,wajah jurnalis dibalut perban saat melaporkan ...
1,1-10,"['7', '15']",Elektabilitas Paslon 02 sebentar lagi terjun b...,Disabilitas,"['0', '0']","['0', '0']","['1', '1']","['0', '0']","['0', '0']","['0', '0']","['0', '0']","['0', '0']","1-10_['7', '15']",Elektabilitas pasangan calon 02 sebentar lagi ...,elektabilitas pasangan calon sebentar lagi ter...
2,1-100,"['7', '15', '32']",Gini aja deh @KPU_ID usul aja nih. Gimana kalo...,Disabilitas,"['0', '0', '0']","['1', '0', '1']","['0', '0', '0']","['0', '0', '0']","['0', '0', '0']","['1', '0', '1']","['0', '0', '0']","['0', '0', '0']","1-100_['7', '15', '32']","Gini aja deh @KPU_ID, usul aja nih. Gimana kal...","gini saja deh , usul saja nih. gimana kalau mi..."
3,1-1000,"['7', '15', '32']",Relawan Tionghoa Kalbar yg di Jakarta mendekla...,Tionghoa,"['0', '0', '0']","['0', '0', '0']","['1', '0', '0']","['0', '0', '0']","['0', '0', '0']","['0', '0', '0']","['0', '0', '0']","['0', '0', '0']","1-1000_['7', '15', '32']",Relawan Tionghoa Kalbar yang di Jakarta mendek...,relawan tionghoa kalbar yang di jakarta mendek...
4,1-1001,"['10', '18']","Kristen, Hindu, Islam dapat perlakuan istimewa...",Kristen,"['0', '0']","['0', '0']","['1', '1']","['0', '0']","['0', '0']","['0', '0']","['0', '0']","['0', '0']","1-1001_['10', '18']","Kristen, Hindu, Islam dapat perlakuan istimewa...","kristen, hindu, islam dapat perlakuan istimewa..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27828,4-995,['19'],Akun WhatsApp Pj Bupati Tegal Agustyarsah,UNKNOWN,['0'],['0'],['0'],['0'],['0'],['0'],['0'],['0'],4-995_['19'],Akun WhatsApp Pj Bupati Tegal Agustyarsah,akun whatsapp pj bupati tegal agustyarsah
27829,4-996,['19'],Buldoser Israel Menabrak Pengungsi Pasien Ruma...,"Jewish, Rohingya",['0'],['0'],['1'],['0'],['0'],['0'],['0'],['0'],4-996_['19'],Bulldozer Israel menabrak pengungsi pasien Rum...,bulldozer israel menabrak pengungsi pasien rum...
27830,4-997,['19'],@TSolihien Sangat perlu di cek dan ricek kemba...,Rohingya,['1'],['0'],['0'],['0'],['0'],['0'],['0'],['0'],4-997_['19'],NaN,NaN
27831,4-998,['19'],Perayaan Chinese New Year tahun ini ya akan ma...,Tionghoa,['0'],['0'],['0'],['0'],['0'],['0'],['0'],['0'],4-998_['19'],Perayaan Chinese New Year tahun ini akan makin...,perayaan chinese new year tahun ini akan makin...


In [4]:
import ast

cols_to_explode = [
    "is_noise_or_spam_text",
    "annotators_id",
    "toxicity",
    "polarized",
    "profanity_obscenity",
    "threat_incitement_to_violence",
    "insults",
    "identity_attack",
    "sexually_explicit"
]

In [5]:
import ast
import pandas as pd

def check_literal(x):
    try:
        ast.literal_eval(x)
        return True
    except:
        return False

df_annotate["is_valid"] = df_annotate["annotators_id"].apply(check_literal)

# Tampilkan data yang error
df_error = df_annotate[~df_annotate["is_valid"]]

df_error[["annotators_id"]]

,annotators_id
5880,"['20', '5', 33']"
5882,"['20', '5', 33']"
10221,"['11', '18', 34']"
10323,"['21', '18', '34]"
10427,"['20', '18', '23', '31', '34]"
25055,"['19', '18', '34]"


In [6]:
import re
def fix_annotators_id(value):
    if pd.isna(value):
        return value

    value = str(value).strip()

    # Coba parse terlebih dahulu
    try:
        result = ast.literal_eval(value)
        return str(result)
    except:
        pass

    # Pastikan dia memang berbentuk [...]
    if value.startswith("["):

        # Ambil semua angka di dalam string
        numbers = re.findall(r"\d+", value)

        # Kalau ada angka, bentuk ulang menjadi list string
        if numbers:
            return str(numbers)

    return value

In [7]:
df_annotate["annotators_id"] = (
    df_annotate["annotators_id"]
    .apply(fix_annotators_id)
)

df_annotate = df_annotate.dropna()

In [8]:
def parse_list(value):
    if pd.isna(value):
        return value

    # Kalau sudah berupa list, tidak perlu diubah
    if isinstance(value, list):
        return value

    # Kalau string, coba parse
    if isinstance(value, str):
        try:
            result = ast.literal_eval(value)

            # Pastikan hasilnya list
            if isinstance(result, list):
                return result

            # Kalau hasilnya bukan list, jadikan list
            return [result]

        except (ValueError, SyntaxError):
            return [value]

    return [value]


for col in cols_to_explode:
    df_annotate[col] = df_annotate[col].apply(parse_list)

In [9]:
print(df_annotate[cols_to_explode].head())

  is_noise_or_spam_text annotators_id   toxicity  polarized  \
0                [0, 0]       [7, 15]     [0, 0]     [0, 0]   
1                [0, 0]       [7, 15]     [0, 0]     [1, 1]   
2             [0, 0, 0]   [7, 15, 32]  [1, 0, 1]  [0, 0, 0]   
3             [0, 0, 0]   [7, 15, 32]  [0, 0, 0]  [1, 0, 0]   
4                [0, 0]      [10, 18]     [0, 0]     [1, 1]   

  profanity_obscenity threat_incitement_to_violence    insults  \
0              [0, 0]                        [0, 0]     [0, 0]   
1              [0, 0]                        [0, 0]     [0, 0]   
2           [0, 0, 0]                     [0, 0, 0]  [1, 0, 1]   
3           [0, 0, 0]                     [0, 0, 0]  [0, 0, 0]   
4              [0, 0]                        [0, 0]     [0, 0]   

  identity_attack sexually_explicit  
0          [0, 0]            [0, 0]  
1          [0, 0]            [0, 0]  
2       [0, 0, 0]         [0, 0, 0]  
3       [0, 0, 0]         [0, 0, 0]  
4          [0, 0]            [0, 0

In [10]:
for col in cols_to_explode:
    print(
        col,
        type(df_annotate[col].iloc[0]),
        df_annotate[col].iloc[0]
    )

is_noise_or_spam_text <class 'list'> ['0', '0']
annotators_id <class 'list'> ['7', '15']
toxicity <class 'list'> ['0', '0']
polarized <class 'list'> ['0', '0']
profanity_obscenity <class 'list'> ['0', '0']
threat_incitement_to_violence <class 'list'> ['0', '0']
insults <class 'list'> ['0', '0']
identity_attack <class 'list'> ['0', '0']
sexually_explicit <class 'list'> ['0', '0']


In [11]:
# Buat tabel pengecekan panjang list
df_check = pd.DataFrame(index=df_annotate.index)

for col in cols_to_explode:
    df_check[col] = df_annotate[col].apply(
        lambda x: len(x) if isinstance(x, (list, tuple)) else 1
    )

# Cari baris yang panjangnya tidak sama
df_mismatch = df_check[
    df_check.nunique(axis=1) > 1
].copy()

print("Jumlah baris mismatch:", len(df_mismatch))

df_mismatch

Jumlah baris mismatch: 34


,is_noise_or_spam_text,annotators_id,toxicity,polarized,profanity_obscenity,threat_incitement_to_violence,insults,identity_attack,sexually_explicit
1622,2,3,2,3,2,2,2,2,2
1696,2,3,3,3,3,2,3,2,2
1698,3,3,3,3,2,3,3,3,3
1714,3,3,3,2,3,3,3,3,3
1834,3,3,2,3,3,3,3,3,3
2374,5,5,4,4,5,4,5,5,5
2395,3,3,3,2,3,3,3,3,3
2427,3,3,3,3,3,2,3,3,3
2613,4,4,4,5,4,5,4,4,4
2616,4,4,5,5,4,5,4,5,4


In [12]:
for idx in df_mismatch.index:
    print(f"\n{'='*80}")
    print(f"INDEX: {idx}")

    for col in cols_to_explode:
        value = df_annotate.loc[idx, col]

        if isinstance(value, (list, tuple)):
            print(f"{col:40} length={len(value)} | {value}")
        else:
            print(f"{col:40} length=1      | {value}")


INDEX: 1622
is_noise_or_spam_text                    length=2 | ['0', '0']
annotators_id                            length=3 | ['9', '8', '32']
toxicity                                 length=2 | ['0', '0']
polarized                                length=3 | ['1', '0', '0']
profanity_obscenity                      length=2 | ['0', '0']
threat_incitement_to_violence            length=2 | ['0', '0']
insults                                  length=2 | ['0', '0']
identity_attack                          length=2 | ['0', '0']
sexually_explicit                        length=2 | ['0', '0']

INDEX: 1696
is_noise_or_spam_text                    length=2 | ['0', '0']
annotators_id                            length=3 | ['9', '8', '32']
toxicity                                 length=3 | ['1', '0', '0']
polarized                                length=3 | ['1', '0', '1']
profanity_obscenity                      length=3 | ['0', '0', '0']
threat_incitement_to_violence            length=2 | ['0', '0

In [13]:
# Baris valid: semua panjang list sama
df_valid = df_annotate[
    ~df_annotate.index.isin(df_mismatch.index)
].copy()

# Baris bermasalah
df_error = df_annotate[
    df_annotate.index.isin(df_mismatch.index)
].copy()

print("Valid   :", len(df_valid))
print("Error   :", len(df_error))

Valid   : 26341
Error   : 34


In [14]:
df_annotate_exp = df_valid.explode(
    cols_to_explode,
    ignore_index=True
)

df_annotate_exp = df_annotate_exp.rename(
    columns={"annotators_id": "annotator_id"}
)

df_annotate_exp["annotator_id"] = df_annotate_exp["annotator_id"].astype("Int64")

df_annotate_exp

,text_id,annotator_id,text,topic,is_noise_or_spam_text,toxicity,polarized,profanity_obscenity,threat_incitement_to_violence,insults,identity_attack,sexually_explicit,key,cleaned_text,processed_text,is_valid
0,1-1,7,Wajah Jurnalis Dibalut Perban Saat Melaporkan ...,UNKNOWN,0,0,0,0,0,0,0,0,"1-1_['7', '15']",Wajah jurnalis dibalut perban saat melaporkan ...,wajah jurnalis dibalut perban saat melaporkan ...,True
1,1-1,15,Wajah Jurnalis Dibalut Perban Saat Melaporkan ...,UNKNOWN,0,0,0,0,0,0,0,0,"1-1_['7', '15']",Wajah jurnalis dibalut perban saat melaporkan ...,wajah jurnalis dibalut perban saat melaporkan ...,True
2,1-10,7,Elektabilitas Paslon 02 sebentar lagi terjun b...,Disabilitas,0,0,1,0,0,0,0,0,"1-10_['7', '15']",Elektabilitas pasangan calon 02 sebentar lagi ...,elektabilitas pasangan calon sebentar lagi ter...,True
3,1-10,15,Elektabilitas Paslon 02 sebentar lagi terjun b...,Disabilitas,0,0,1,0,0,0,0,0,"1-10_['7', '15']",Elektabilitas pasangan calon 02 sebentar lagi ...,elektabilitas pasangan calon sebentar lagi ter...,True
4,1-100,7,Gini aja deh @KPU_ID usul aja nih. Gimana kalo...,Disabilitas,0,1,0,0,0,1,0,0,"1-100_['7', '15', '32']","Gini aja deh @KPU_ID, usul aja nih. Gimana kal...","gini saja deh , usul saja nih. gimana kalau mi...",True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
51965,4-994,19,@pippsstudy abis packing nnt ingfokan cowo kri...,Kristen,0,0,0,0,0,0,0,0,4-994_['19'],"@pippsstudy Abis packing, nanti infokan cowok ...","abis packing, nanti infokan cowok kristen.",True
51966,4-995,19,Akun WhatsApp Pj Bupati Tegal Agustyarsah,UNKNOWN,0,0,0,0,0,0,0,0,4-995_['19'],Akun WhatsApp Pj Bupati Tegal Agustyarsah,akun whatsapp pj bupati tegal agustyarsah,True
51967,4-996,19,Buldoser Israel Menabrak Pengungsi Pasien Ruma...,"Jewish, Rohingya",0,0,1,0,0,0,0,0,4-996_['19'],Bulldozer Israel menabrak pengungsi pasien Rum...,bulldozer israel menabrak pengungsi pasien rum...,True
51968,4-998,19,Perayaan Chinese New Year tahun ini ya akan ma...,Tionghoa,0,0,0,0,0,0,0,0,4-998_['19'],Perayaan Chinese New Year tahun ini akan makin...,perayaan chinese new year tahun ini akan makin...,True


In [15]:
df_annotator = pd.read_csv("../model with metadata1/dataset/annotator_demographic.csv")
df_annotator

,annotator_id,ethnicity,religion,disability,lgbt,gender,age,domisili,pendidikan terakhir,status pekerjaan,president vote leaning
0,1,Tionghoa,Kristen,no,no,F,50,Jakarta,Diploma,Bekerja,2
1,2,Madura,Islam,no,no,F,22,Madura,Sekolah Menengah Atas (SMA),Pelajar/Mahasiswa,3
2,3,Batak,Kepercayaan Lokal,no,no,F,29,Medan,Sarjana (S1),Bekerja,3
3,4,Bali,Hindu,no,no,M,23,Bandung,Sekolah Menengah Atas (SMA),Pelajar/Mahasiswa,-
4,5,Jawa,Syiah,no,no,M,46,Bandung,Diploma,Bekerja,Golput
5,6,Minang,Ahmadiyah,no,no,F,22,Jakarta,Sarjana (S1),Bekerja,3
6,7,Minang,Islam,yes,no,F,27,Padang,Sarjana (S1),Bekerja,1
7,8,Arab,Islam,no,no,M,23,Jakarta,Sekolah Menengah Atas (SMA),Pelajar/Mahasiswa,1
8,9,Jawa,Islam,no,no,F,21,Magelang,Sekolah Menengah Atas (SMA),Pelajar/Mahasiswa,2
9,10,Tionghoa,Buddha,no,no,F,20,Tangerang,Sekolah Menengah Atas (SMA),Pelajar/Mahasiswa,3


In [16]:
df_annotate_exp.info()

<class 'pandas.DataFrame'>
RangeIndex: 51970 entries, 0 to 51969
Data columns (total 16 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   text_id                        51970 non-null  str  
 1   annotator_id                   51970 non-null  Int64
 2   text                           51970 non-null  str  
 3   topic                          51970 non-null  str  
 4   is_noise_or_spam_text          51970 non-null  str  
 5   toxicity                       51970 non-null  str  
 6   polarized                      51970 non-null  str  
 7   profanity_obscenity            51970 non-null  str  
 8   threat_incitement_to_violence  51970 non-null  str  
 9   insults                        51970 non-null  str  
 10  identity_attack                51970 non-null  str  
 11  sexually_explicit              51970 non-null  str  
 12  key                            51970 non-null  str  
 13  cleaned_text               

In [17]:
df_annotator.info()

<class 'pandas.DataFrame'>
RangeIndex: 32 entries, 0 to 31
Data columns (total 11 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   annotator_id            32 non-null     int64
 1   ethnicity               32 non-null     str  
 2   religion                32 non-null     str  
 3   disability              32 non-null     str  
 4   lgbt                    32 non-null     str  
 5   gender                  32 non-null     str  
 6   age                     32 non-null     int64
 7   domisili                32 non-null     str  
 8   pendidikan terakhir     32 non-null     str  
 9   status pekerjaan        32 non-null     str  
 10  president vote leaning  32 non-null     str  
dtypes: int64(2), str(9)
memory usage: 4.5 KB


In [18]:
annotate_columns = df_annotate_exp.drop(['annotator_id', 'text_id'], axis=1).columns.to_list()
annotator_columns = df_annotator.drop(['annotator_id'], axis=1).columns.to_list()

print(annotate_columns)
print(annotator_columns)

['text', 'topic', 'is_noise_or_spam_text', 'toxicity', 'polarized', 'profanity_obscenity', 'threat_incitement_to_violence', 'insults', 'identity_attack', 'sexually_explicit', 'key', 'cleaned_text', 'processed_text', 'is_valid']
['ethnicity', 'religion', 'disability', 'lgbt', 'gender', 'age', 'domisili', 'pendidikan terakhir', 'status pekerjaan', 'president vote leaning']


In [19]:
df = df_annotate_exp.merge(
    df_annotator,
    on="annotator_id",
    how="left"
)
df = df.drop(['text_id', 'annotator_id'], axis=1)
df

,text,topic,is_noise_or_spam_text,toxicity,polarized,profanity_obscenity,threat_incitement_to_violence,insults,identity_attack,sexually_explicit,...,ethnicity,religion,disability,lgbt,gender,age,domisili,pendidikan terakhir,status pekerjaan,president vote leaning
0,Wajah Jurnalis Dibalut Perban Saat Melaporkan ...,UNKNOWN,0,0,0,0,0,0,0,0,...,Minang,Islam,yes,no,F,27,Padang,Sarjana (S1),Bekerja,1
1,Wajah Jurnalis Dibalut Perban Saat Melaporkan ...,UNKNOWN,0,0,0,0,0,0,0,0,...,Tionghoa,Katolik,yes,no,F,21,Jakarta,Sekolah Menengah Atas (SMA),Pelajar/Mahasiswa,3
2,Elektabilitas Paslon 02 sebentar lagi terjun b...,Disabilitas,0,0,1,0,0,0,0,0,...,Minang,Islam,yes,no,F,27,Padang,Sarjana (S1),Bekerja,1
3,Elektabilitas Paslon 02 sebentar lagi terjun b...,Disabilitas,0,0,1,0,0,0,0,0,...,Tionghoa,Katolik,yes,no,F,21,Jakarta,Sekolah Menengah Atas (SMA),Pelajar/Mahasiswa,3
4,Gini aja deh @KPU_ID usul aja nih. Gimana kalo...,Disabilitas,0,1,0,0,0,1,0,0,...,Minang,Islam,yes,no,F,27,Padang,Sarjana (S1),Bekerja,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
51965,@pippsstudy abis packing nnt ingfokan cowo kri...,Kristen,0,0,0,0,0,0,0,0,...,Batak,Kristen,no,no,F,39,Depok,Magister (S2),Bekerja,2
51966,Akun WhatsApp Pj Bupati Tegal Agustyarsah,UNKNOWN,0,0,0,0,0,0,0,0,...,Batak,Kristen,no,no,F,39,Depok,Magister (S2),Bekerja,2
51967,Buldoser Israel Menabrak Pengungsi Pasien Ruma...,"Jewish, Rohingya",0,0,1,0,0,0,0,0,...,Batak,Kristen,no,no,F,39,Depok,Magister (S2),Bekerja,2
51968,Perayaan Chinese New Year tahun ini ya akan ma...,Tionghoa,0,0,0,0,0,0,0,0,...,Batak,Kristen,no,no,F,39,Depok,Magister (S2),Bekerja,2


In [20]:
df["gender"] = df["gender"].str.strip()

In [21]:
df.to_csv("dataset_exploded.csv")

# Applying Metadata

In [22]:
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict

c:\Users\GY24-2\AppData\Local\anaconda3\envs\thesis\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
